In [2]:
import os
from dotenv import load_dotenv
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_openai import OpenAIEmbeddings
from langchain.schema import Document
from langchain_community.vectorstores import Chroma

import numpy as np
from typing import List


load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

# 1. Document Loading

In [3]:
loader = DirectoryLoader(
    path='../data/txt',
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding":"utf-8"},
    show_progress=True
)
documents = loader.load()

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 79.44it/s]


# 2. Chunking

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(documents=documents)

len(chunks)

34

# 3. Apply Embedding to the chunks and store in Vector Store

In [7]:
store_location = "../vector_store/chroma_db"
embeddings_model = OpenAIEmbeddings(model='text-embedding-3-small')
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model,
    persist_directory=store_location,
    collection_name="basic_rag_collection"
)

print(f"Number of vectors created: {vector_store._collection.count()}")

Number of vectors created: 34


# 4. Perform Similarity Search

In [8]:
query = """What are some necessary documents to purchase land in India?"""

In [12]:
similar_docs = vector_store.similarity_search(query=query, k=3)

In [21]:
for idx,doc in enumerate(similar_docs):
    print(f"Doc {idx+1}: {doc.page_content}")
    print("\n")
    print(f'Source: {doc.metadata.get("source", "unknown")}')

Doc 1: When buying farmland in India, you must check land title deeds and encumbrance certificates to ensure clear ownership and no financial liabilities, along with revenue records like the Record of Rights


Source: ..\data\txt\india real estate checklist.txt
Doc 2: To buy farmland in Karnataka, you must gather essential documents including the seller's ID and PAN cards, Title Deed, Sale Deed, Encumbrance Certificate, Property Tax receipts, and Record of Rights


Source: ..\data\txt\karnataka document checklist.txt
Doc 3: Conversion Certificate: If the land's use has been changed from agricultural to non-agricultural, this certificate is required. 
Buyer & Seller Identification:


Source: ..\data\txt\karnataka document checklist.txt


# Similarity Search with Scores

In [23]:
results_with_scores = vector_store.similarity_search_with_score(query=query, k=3)

In [27]:
for idx, doc in enumerate(results_with_scores):
    print(f"Doc {idx+1}: Score: {doc[1]:.3f}")

Doc 1: Score: 0.619
Doc 2: Score: 0.667
Doc 3: Score: 0.870


# 